# TensorBinding — GPU vs CPU correctness tests

Validates that all GPU-enabled functions (`run_on=:gpu`) produce numerically identical results to the CPU reference (`run_on=:cpu`).

Each section builds a small tight-binding Hamiltonian (L=4, N=16 sites) and compares the relative error ‖CPU − GPU‖/‖CPU‖ across five families of functions:

| Section | Functions tested |
|---------|------------------|
| 1 — KPM | `KPM_Tn`, `get_ldos_online`, `get_dos_stochastic` |
| 2 — Bands | `get_bands` |
| 3 — Purification | `mcweeny_purify`, `sp2_purify`, `get_density` |
| 4 — Topology | `get_W`, `get_C` |
| 5 — Time evolution | `evolve_rk4_dm_timedep` |

The GPU path requires CUDA.jl in the active Julia environment.  If no GPU is detected (`CUDA.functional() == false`) the GPU assertions are skipped and a message is printed instead.

Note: Since this is a Julia notebook, the first cell execution includes function precompilation overhead.

In [23]:
using LinearAlgebra
using Plots
using LaTeXStrings
using ITensors
using ITensorMPS
using CUDA
using NDTensors
include("../src/TensorBinding.jl")
using .TensorBinding

# Since TensorBinding is loaded via include() rather than as a registered package,
# the Julia 1.9 extension mechanism (ext/TensorBindingCUDAExt.jl) is not triggered
# automatically.  Extend the GPU dispatch stubs manually.
TensorBinding.to_device(x::MPO, ::TensorBinding.GPUBackend) = NDTensors.cu(x)
TensorBinding.to_device(x::MPS, ::TensorBinding.GPUBackend) = NDTensors.cu(x)
TensorBinding.from_device(x::MPO, ::TensorBinding.GPUBackend) = NDTensors.cpu(x)
TensorBinding.from_device(x::MPS, ::TensorBinding.GPUBackend) = NDTensors.cpu(x)

has_gpu = CUDA.functional()
println("CUDA available: ", has_gpu)
has_gpu && println("GPU device:     ", CUDA.name(CUDA.device()))

CUDA available: true
GPU device:     NVIDIA GeForce RTX 3050 Ti Laptop GPU


---
## Section 1 — Kernel Polynomial Method (KPM)

Three KPM routines are tested on a 1D tight-binding chain with L=4 (N=16 sites).

- **`KPM_Tn`**: builds the Chebyshev MPO sequence $T_0(\tilde{H}), \ldots, T_N(\tilde{H})$ via the three-term recurrence.  CPU and GPU sequences are compared by taking the norm of the difference of the last Chebyshev MPO.
- **`get_ldos_online`**: computes the local density of states at a single site streaming Chebyshev moments without storing the full Tn cache.
- **`get_dos_stochastic`**: stochastic trace estimator for the global DOS.

In [24]:
# ── Shared KPM setup ──────────────────────────────────────────────────────────
L_kpm  = 20
Ncheb  = 50
ω_list = range(-3.0, 3.0; length=60) |> collect

H_kpm_cpu = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_kpm, scale=4.5)
println(H_kpm_cpu)

TBHamiltonian | L=20, N=1048576, scale=4.5, maxlinkdim=3 | geometry: 1048576 sites, 1D | no Tn cache


In [26]:
# ── KPM_Tn: Chebyshev MPO sequence with timing ────────────────────────────
# Single Hamiltonian for both runs so the Tn MPOs share site indices.
H_kpm = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_kpm, scale=4.5)

t_kpm_cpu = @elapsed begin
    Tn_cpu, _, _ = TensorBinding.KPM_Tn(
        H_kpm, Ncheb; maxdim=30, run_on=:cpu, verbose=false)
end
println("KPM_Tn (CPU): ", round(t_kpm_cpu; digits=2), " s",
        "  maxlinkdim(T_$(Ncheb)) = ", ITensorMPS.maxlinkdim(Tn_cpu[end]))

if has_gpu
    t_kpm_gpu = @elapsed begin
        Tn_gpu, _, _ = TensorBinding.KPM_Tn(
            H_kpm, Ncheb; maxdim=30, run_on=:gpu, verbose=false)
    end
    println("KPM_Tn (GPU): ", round(t_kpm_gpu; digits=2), " s")
    println("Speedup:       ", round(t_kpm_cpu / t_kpm_gpu; digits=2), "×")

    diff_mpo = +(Tn_cpu[end], (-1.0) * Tn_gpu[end]; cutoff=1e-15)
    rel_err  = norm(diff_mpo) / norm(Tn_cpu[end])
    println("‖T_$(Ncheb)(CPU) − T_$(Ncheb)(GPU)‖ / ‖T_$(Ncheb)‖ = ", rel_err)
    @assert rel_err < 1e-4  "KPM_Tn CPU/GPU mismatch"
    println("  ✓  KPM_Tn CPU and GPU agree")
else
    println("No GPU detected — KPM_Tn GPU comparison skipped")
end

KPM_Tn (CPU): 1.32 s  maxlinkdim(T_50) = 7
KPM_Tn (GPU): 27.54 s
Speedup:       0.05×
‖T_50(CPU) − T_50(GPU)‖ / ‖T_50‖ = 5.10630653787099e-7
  ✓  KPM_Tn CPU and GPU agree


In [ ]:
# ── KPM_Tn: AAH + modulated 3rd-neighbor hopping (bond-dim stress test) ────
# chain_1d saturates at bond dim ~7 — too small for GPU to overcome launch overhead.
# AAH (quasiperiodic on-site) + long-range t3 hopping forces larger bond dims.
L_aah      = 20        # 128 sites
Ncheb_aah  = 80
maxdim_aah = 150

b_gold  = (1 + sqrt(5)) / 2    # incommensurate frequency
H_aah   = TensorBinding.get_Hamiltonian("aah", (V=2.0, phi=0.0, t=1.0); L=L_aah)
TensorBinding.add_hopping!(H_aah, i -> 0.3 * cos(2π * b_gold * i + π/3); nn=3, maxdim=20)
H_aah.scale = 5.5   # AAH(V=2,t=1) + t3=0.3: bandwidth ~4.6, padded

t_aah_cpu = @elapsed begin
    Tn_aah_cpu, _, _ = TensorBinding.KPM_Tn(
        H_aah, Ncheb_aah; maxdim=maxdim_aah, run_on=:cpu, verbose=false)
end
println("KPM_Tn AAH+t3 (CPU): ", round(t_aah_cpu; digits=2), " s",
        "  maxlinkdim(T_$(Ncheb_aah)) = ", ITensorMPS.maxlinkdim(Tn_aah_cpu[end]))

if has_gpu
    t_aah_gpu = @elapsed begin
        Tn_aah_gpu, _, _ = TensorBinding.KPM_Tn(
            H_aah, Ncheb_aah; maxdim=maxdim_aah, run_on=:gpu, verbose=false)
    end
    println("KPM_Tn AAH+t3 (GPU): ", round(t_aah_gpu; digits=2), " s")
    println("Speedup:              ", round(t_aah_cpu / t_aah_gpu; digits=2), "×")

    diff_mpo = +(Tn_aah_cpu[end], (-1.0) * Tn_aah_gpu[end]; cutoff=1e-15)
    rel_err  = norm(diff_mpo) / norm(Tn_aah_cpu[end])
    println("‖T_$(Ncheb_aah)(CPU) − T_$(Ncheb_aah)(GPU)‖ / ‖T_$(Ncheb_aah)‖ = ", rel_err)
    @assert rel_err < 1e-5  "KPM_Tn AAH+t3 CPU/GPU mismatch"
    println("  ✓  KPM_Tn AAH+t3 CPU and GPU agree")
else
    println("No GPU detected — KPM_Tn AAH+t3 GPU comparison skipped")
end

In [20]:
# ── get_ldos_online: site-resolved LDOS with timing ────────────────────────
X_site = 2^(L_kpm - 1)   # middle site

H_ldos_cpu = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_kpm, scale=4.5)
t_ldos_cpu = @elapsed begin
    ldos_cpu = TensorBinding.get_ldos_online(
        H_ldos_cpu, Ncheb, X_site, ω_list; maxdim=30, run_on=:cpu, verbose=false)
end
println("get_ldos_online (CPU): ", round(t_ldos_cpu; digits=2), " s",
        "  max LDOS = ", round(maximum(ldos_cpu); digits=4))

if has_gpu
    H_ldos_gpu = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_kpm, scale=4.5)
    t_ldos_gpu = @elapsed begin
        ldos_gpu = TensorBinding.get_ldos_online(
            H_ldos_gpu, Ncheb, X_site, ω_list; maxdim=30, run_on=:gpu, verbose=false)
    end
    println("get_ldos_online (GPU): ", round(t_ldos_gpu; digits=2), " s")
    println("Speedup:                ", round(t_ldos_cpu / t_ldos_gpu; digits=2), "×")

    rel_err = norm(ldos_cpu - ldos_gpu) / norm(ldos_cpu)
    println("‖CPU − GPU‖ / ‖CPU‖ = ", rel_err)
    @assert rel_err < 1e-5  "get_ldos_online CPU/GPU mismatch"
    println("  ✓  get_ldos_online CPU and GPU agree")
else
    println("No GPU detected — get_ldos_online GPU comparison skipped")
end

get_ldos_online (CPU): 1.03 s  max LDOS = 0.4786
get_ldos_online (GPU): 5.24 s
Speedup:                0.2×
‖CPU − GPU‖ / ‖CPU‖ = 3.845696588812885e-15
  ✓  get_ldos_online CPU and GPU agree


In [ ]:
# ── LDOS plot ────────────────────────────────────────────────────────────────
p_ldos = plot(ω_list, ldos_cpu;
              xlabel=L"ω", ylabel=L"\mathrm{LDOS}(\omega)",
              title="LDOS at site $(X_site)  ($(2^L_kpm)-site chain, Ncheb=$(Ncheb))",
              label="CPU", color=:steelblue, lw=2, framestyle=:box)
if has_gpu
    plot!(p_ldos, ω_list, ldos_gpu;
          label="GPU", color=:firebrick, ls=:dash, lw=1.5, alpha=0.8)
end
p_ldos

In [9]:
# ── get_dos_stochastic: stochastic DOS with timing ───────────────────────────
H_dos_cpu = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_kpm, scale=4.5)
t_dos_cpu = @elapsed begin
    dos_cpu = TensorBinding.get_dos_stochastic(
        H_dos_cpu, Ncheb, ω_list; N_sample=8, seed=42, maxdim=30, run_on=:cpu)
end
println("get_dos_stochastic (CPU): ", round(t_dos_cpu; digits=2), " s",
        "  max DOS = ", round(maximum(dos_cpu); digits=4))

if has_gpu
    H_dos_gpu = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_kpm, scale=4.5)
    t_dos_gpu = @elapsed begin
        dos_gpu = TensorBinding.get_dos_stochastic(
            H_dos_gpu, Ncheb, ω_list; N_sample=8, seed=42, maxdim=30, run_on=:gpu)
    end
    println("get_dos_stochastic (GPU): ", round(t_dos_gpu; digits=2), " s")
    println("Speedup:                   ", round(t_dos_cpu / t_dos_gpu; digits=2), "×")

    rel_err = norm(dos_cpu - dos_gpu) / norm(dos_cpu)
    println("‖CPU − GPU‖ / ‖CPU‖ = ", rel_err)
    @assert rel_err < 1e-5  "get_dos_stochastic CPU/GPU mismatch"
    println("  ✓  get_dos_stochastic CPU and GPU agree")
else
    println("No GPU detected — get_dos_stochastic GPU comparison skipped")
end

get_dos_stochastic (CPU): max DOS = 5.082590099275335
get_dos_stochastic: ‖CPU − GPU‖ / ‖CPU‖ = 5.768568502591049e-16
  ✓  get_dos_stochastic CPU and GPU agree


In [ ]:
# ── DOS plot ─────────────────────────────────────────────────────────────────
p_dos = plot(ω_list, dos_cpu;
             xlabel=L"ω", ylabel=L"\mathrm{DOS}(\omega)",
             title="Stochastic DOS  (8 random vectors, $(2^L_kpm)-site chain)",
             label="CPU", color=:steelblue, lw=2, framestyle=:box)
if has_gpu
    plot!(p_dos, ω_list, dos_gpu;
          label="GPU", color=:firebrick, ls=:dash, lw=1.5, alpha=0.8)
end
p_dos

---
## Section 2 — Band structure via QFT (`get_bands`)

`get_bands` applies an inverse QFT to the Chebyshev MPOs to extract the k-resolved spectral function $A(k,\omega)$.  The function internally calls `KPM_Tn` and `apply` — both GPU-enabled.  The resulting spectral matrices (shape: `n_omega × n_k`) are compared element-wise.

A 1D chain with L=4 is used; `num_x=16` gives one k-point per site.

In [12]:
# ── get_bands: k-resolved spectral function with timing ──────────────────────
L_bands = 4
e_phys  = range(-3.0, 3.0; length=60) |> collect

H_bands_cpu = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_bands, scale=4.5)
t_bands_cpu = @elapsed begin
    Ak_cpu = TensorBinding.get_bands(
        H_bands_cpu, 20, 1, e_phys; num_x=2^L_bands, maxdim=50, run_on=:cpu)
end
println("get_bands (CPU): ", round(t_bands_cpu; digits=2), " s",
        "  size(Ak) = ", size(Ak_cpu), "  max = ", round(maximum(Ak_cpu); digits=4))

if has_gpu
    H_bands_gpu = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_bands, scale=4.5)
    t_bands_gpu = @elapsed begin
        Ak_gpu = TensorBinding.get_bands(
            H_bands_gpu, 20, 1, e_phys; num_x=2^L_bands, maxdim=50, run_on=:gpu)
    end
    println("get_bands (GPU): ", round(t_bands_gpu; digits=2), " s")
    println("Speedup:          ", round(t_bands_cpu / t_bands_gpu; digits=2), "×")

    rel_err = norm(Ak_cpu - Ak_gpu) / norm(Ak_cpu)
    println("‖CPU − GPU‖ / ‖CPU‖ = ", rel_err)
    @assert rel_err < 1e-5  "get_bands CPU/GPU mismatch"
    println("  ✓  get_bands CPU and GPU agree")
else
    println("No GPU detected — get_bands GPU comparison skipped")
end

get_bands (CPU): size(Ak) = (60, 16),  max = 0.8703797663322151
get_bands: ‖CPU − GPU‖ / ‖CPU‖ = 5.572160412408631e-15
  ✓  get_bands CPU and GPU agree


In [ ]:
# ── Bands plot ───────────────────────────────────────────────────────────────
kvals  = (2π / 2^L_bands) .* (0:2^L_bands-1) .- π

# Analytic dispersion: E(k) = -2t cos(k) for a 1D chain
k_fine  = range(-π, π; length=300) |> collect
E_exact = -2.0 .* cos.(k_fine)

vmax    = maximum(Ak_cpu)
p_bands = heatmap(kvals, e_phys, Ak_cpu;
                  xlabel=L"k", ylabel=L"E",
                  title=L"A(k,\omega) \ \mathrm{1D\ chain\ (CPU)}",
                  color=:inferno, clims=(0, vmax), flipy=true,
                  framestyle=:box, size=(600, 420))
plot!(p_bands, k_fine, E_exact;
      label=L"-2t\cos k", color=:white, ls=:dash, lw=1.2, alpha=0.85)
p_bands

---
## Section 3 — Density matrix purification

Three purification paths are tested:

- **`mcweeny_purify`**: $\rho_{n+1} = 3\rho_n^2 - 2\rho_n^3$, quadratic convergence, 2 MPO products per step.
- **`sp2_purify`**: $\rho_{n+1} = \rho_n^2$ or $2\rho_n - \rho_n^2$, 1 MPO product per step.
- **`get_density`** with `method=:kpm`: uses the cached Chebyshev MPOs.

All three start from the same initial guess `ρ₀ = (I − H/scale)/2` and should converge to the same projector $P = \theta(\epsilon_F - H)$. CPU and GPU results are compared via ‖ρ_CPU − ρ_GPU‖/‖ρ_CPU‖.

In [ ]:
# ── Shared purification setup ─────────────────────────────────────────────────
L_pur = 4
H_pur = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_pur, scale=4.5)
println(H_pur)

# Initial guess: ρ₀ = (I − H/scale)/2, spectrum ⊆ [0, 1]
ρ0 = TensorBinding.purification_initial_guess(H_pur; maxdim=30)
println("ρ₀ maxlinkdim = ", ITensorMPS.maxlinkdim(ρ0))

In [ ]:
# ── mcweeny_purify ────────────────────────────────────────────────────────────
ρ_mcw_cpu = TensorBinding.mcweeny_purify(ρ0; maxdim=30, tol=1e-5, verbose=true, run_on=:cpu)
println("mcweeny_purify (CPU): maxlinkdim = ", ITensorMPS.maxlinkdim(ρ_mcw_cpu))

if has_gpu
    ρ_mcw_gpu = TensorBinding.mcweeny_purify(ρ0; maxdim=30, tol=1e-5, verbose=false, run_on=:gpu)

    diff_rho = +(ρ_mcw_cpu, (-1.0) * ρ_mcw_gpu; cutoff=1e-15)
    rel_err  = norm(diff_rho) / norm(ρ_mcw_cpu)
    println("mcweeny_purify: ‖ρ_CPU − ρ_GPU‖ / ‖ρ_CPU‖ = ", rel_err)
    @assert rel_err < 1e-4  "mcweeny_purify CPU/GPU mismatch"
    println("  ✓  mcweeny_purify CPU and GPU agree")
else
    println("No GPU detected — mcweeny_purify GPU comparison skipped")
end

In [ ]:
# ── sp2_purify ────────────────────────────────────────────────────────────────
Nel = H_pur.N ÷ 2

ρ_sp2_cpu = TensorBinding.sp2_purify(ρ0, Nel; maxdim=30, tol=1e-5, verbose=true, run_on=:cpu)
println("sp2_purify (CPU): maxlinkdim = ", ITensorMPS.maxlinkdim(ρ_sp2_cpu))

if has_gpu
    ρ_sp2_gpu = TensorBinding.sp2_purify(ρ0, Nel; maxdim=30, tol=1e-5, verbose=false, run_on=:gpu)

    diff_rho = +(ρ_sp2_cpu, (-1.0) * ρ_sp2_gpu; cutoff=1e-15)
    rel_err  = norm(diff_rho) / norm(ρ_sp2_cpu)
    println("sp2_purify: ‖ρ_CPU − ρ_GPU‖ / ‖ρ_CPU‖ = ", rel_err)
    @assert rel_err < 1e-4  "sp2_purify CPU/GPU mismatch"
    println("  ✓  sp2_purify CPU and GPU agree")
else
    println("No GPU detected — sp2_purify GPU comparison skipped")
end

In [ ]:
# ── get_density: unified wrapper — all three methods ─────────────────────────
# Use one Hamiltonian per method iteration; clear caches before the GPU run so
# it re-computes from scratch on the same H (same site indices as the CPU run).
for method in [:mcweeny, :sp2, :kpm]
    H = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_pur, scale=4.5)

    ρ_cpu = TensorBinding.get_density(H; method=method, maxdim=30, Ncheb=Ncheb,
                                       tol=1e-5, run_on=:cpu)

    if has_gpu
        # Clear caches so the GPU call re-runs (rather than returning the cached CPU result)
        H._density_cache = nothing
        H._tn_cache      = nothing

        ρ_gpu = TensorBinding.get_density(H; method=method, maxdim=30, Ncheb=Ncheb,
                                           tol=1e-5, run_on=:gpu)

        diff_rho = +(ρ_cpu, (-1.0) * ρ_gpu; cutoff=1e-15)
        rel_err  = norm(diff_rho) / norm(ρ_cpu)
        println("get_density(:$(method)): ‖ρ_CPU − ρ_GPU‖ / ‖ρ_CPU‖ = ", rel_err)
        @assert rel_err < 1e-4  "get_density(:$(method)) CPU/GPU mismatch"
        println("  ✓  get_density(:$(method)) CPU and GPU agree")
    else
        println("get_density(:$(method)) — no GPU detected, skipping comparison")
    end
end

---
## Section 4 — Topological invariants

Two topological observables are tested:

- **`get_W`** — 1D winding number on an SSH chain.  The winding number $W = \sum_\alpha \langle \alpha | \hat{W} | \alpha \rangle$ sums matrix elements over all $2^L$ computational basis states using the GPU-resident $\hat{W}$ MPO and `inner` calls where the basis states are moved to the device.
- **`get_C`** — real-space Chern marker $C(r)$ on a 2D honeycomb lattice (L=4, Lx=2, Ly=2). The same device-resident projector and position MPOs are used throughout.

The comparison checks that CPU and GPU return the same winding/Chern values to within numerical tolerance.

In [ ]:
# ── get_W: SSH winding number ─────────────────────────────────────────────
L_ssh = 4
H_ssh_cpu = TensorBinding.get_Hamiltonian("ssh", Dict(:t => 1, :d => 0.5); L=L_ssh, scale=5.5)
H_ssh_gpu = TensorBinding.get_Hamiltonian("ssh", Dict(:t => 1, :d => 0.5); L=L_ssh, scale=5.5)

# x-coordinate: site index (0-indexed)
xfunc_ssh  = (i, Lc) -> Float64(i)
# sz-function: sublattice sign (alternating +/-1)
sz_func_ssh = (i, Lc) -> Float64((-1)^i)

# get_W returns a closure calculate_winding(alpha::Int) -> ComplexF64 (1-indexed)
calc_W_cpu = TensorBinding.get_W(
    H_ssh_cpu, xfunc_ssh, sz_func_ssh;
    method=:KPM, Nchebychev=30, maxdim=20, run_on=:cpu)
winding_cpu = real(sum(calc_W_cpu(alpha) for alpha in 1:2^L_ssh))
println("get_W (CPU): winding number = ", winding_cpu)

if has_gpu
    calc_W_gpu = TensorBinding.get_W(
        H_ssh_gpu, xfunc_ssh, sz_func_ssh;
        method=:KPM, Nchebychev=30, maxdim=20, run_on=:gpu)
    winding_gpu = real(sum(calc_W_gpu(alpha) for alpha in 1:2^L_ssh))

    err = abs(winding_cpu - winding_gpu)
    println("get_W: |W_CPU − W_GPU| = ", err)
    @assert err < 1e-4  "get_W CPU/GPU mismatch"
    println("  ✓  get_W CPU and GPU agree")
else
    println("No GPU detected — get_W GPU comparison skipped")
end

In [ ]:
# ── get_C: real-space Chern marker on a 2D honeycomb ─────────────────
# L=4, Lx=2, Ly=2: 2x2 honeycomb unit cell (2^4 = 16 quantics sites).
L_hc = 4; Lx_hc = 2; Ly_hc = L_hc - Lx_hc

H_hc_cpu = TensorBinding.get_Hamiltonian("honeycomb", 1.0; L=L_hc, Lx=Lx_hc, scale=5.5)
H_hc_gpu = TensorBinding.get_Hamiltonian("honeycomb", 1.0; L=L_hc, Lx=Lx_hc, scale=5.5)

Nx_hc = 2^Lx_hc   # x-sites

# Site index i (0-indexed) encoded as ix + Nx*iy in quantics row-major
xfunc_hc = (i, Lc) -> Float64(i % Nx_hc)
yfunc_hc = (i, Lc) -> Float64(i ÷ Nx_hc)

# get_C returns a closure calculate_chern_number(alpha::Int) -> ComplexF64 (1-indexed)
calc_C_cpu = TensorBinding.get_C(
    H_hc_cpu, xfunc_hc, yfunc_hc;
    method=:KPM, Nchebychev=30, maxdim=20, run_on=:cpu)
chern_cpu = real(sum(calc_C_cpu(alpha) for alpha in 1:2^H_hc_cpu.L))
println("get_C (CPU): Chern number = ", chern_cpu)

if has_gpu
    calc_C_gpu = TensorBinding.get_C(
        H_hc_gpu, xfunc_hc, yfunc_hc;
        method=:KPM, Nchebychev=30, maxdim=20, run_on=:gpu)
    chern_gpu = real(sum(calc_C_gpu(alpha) for alpha in 1:2^H_hc_gpu.L))

    err = abs(chern_cpu - chern_gpu)
    println("get_C: |C_CPU − C_GPU| = ", err)
    @assert err < 1e-4  "get_C CPU/GPU mismatch"
    println("  ✓  get_C CPU and GPU agree")
else
    println("No GPU detected — get_C GPU comparison skipped")
end

---
## Section 5 — Time evolution (`evolve_rk4_dm_timedep`)

`evolve_rk4_dm_timedep` propagates a density matrix under a time-dependent Hamiltonian using a 4th-order Runge–Kutta scheme:

$$\rho(t + \mathrm{d}t) = \rho(t) + \tfrac{1}{6}(k_1 + 2k_2 + 2k_3 + k_4)\,\mathrm{d}t$$

The density matrix $\rho$ lives on the GPU throughout the loop; only the stored snapshots in `states` are moved back to CPU via `from_device`.  The factory `Hoft(t)` always returns a CPU MPO and is moved to the device inside each `rk4_step_dm_timedep` call.

A 1D chain is used with a time-independent $H(t) = H$ (simplest non-trivial case).  CPU and GPU state trajectories are compared step-by-step.

In [ ]:
# ── evolve_rk4_dm_timedep ────────────────────────────────────────────────────
L_ev  = 4
H_ev  = TensorBinding.get_Hamiltonian("chain_1d", 1.0; L=L_ev, scale=4.5)

# Time-independent factory: Hoft(t) = H.mpo  (always returns a CPU MPO)
Hoft = t -> H_ev.mpo

# Initial density matrix: projector onto the lower half of the spectrum
ρ0_ev = TensorBinding.purification_initial_guess(H_ev; maxdim=20)
println("ρ₀ maxlinkdim = ", ITensorMPS.maxlinkdim(ρ0_ev))

nsteps = 5
dt     = 0.05

states_cpu = TensorBinding.evolve_rk4_dm_timedep(
    Hoft, ρ0_ev, nsteps, dt;
    maxdim=30, cutoff=1e-10, verbose=true, run_on=:cpu)
println("evolve_rk4_dm_timedep (CPU): $(length(states_cpu)) snapshots")

if has_gpu
    states_gpu = TensorBinding.evolve_rk4_dm_timedep(
        Hoft, ρ0_ev, nsteps, dt;
        maxdim=30, cutoff=1e-10, verbose=false, run_on=:gpu)

    errs = Float64[]
    for k in 1:nsteps+1
        diff_k  = +(states_cpu[k], (-1.0) * states_gpu[k]; cutoff=1e-15)
        rel_k   = norm(diff_k) / norm(states_cpu[k])
        push!(errs, rel_k)
    end
    println("evolve_rk4_dm_timedep: max ‖ρ_CPU − ρ_GPU‖/‖ρ_CPU‖ over all steps = ",
            maximum(errs))
    @assert maximum(errs) < 1e-4  "evolve_rk4_dm_timedep CPU/GPU mismatch"
    println("  ✓  evolve_rk4_dm_timedep CPU and GPU agree on all snapshots")
else
    println("No GPU detected — evolve_rk4_dm_timedep GPU comparison skipped")
end

---
## Summary

All GPU-enabled functions in TensorBinding have been tested.  If all cells ran without assertion errors the GPU and CPU backends produce numerically equivalent results (relative error < 10⁻⁴ in all cases).

| Function | GPU-enabled argument | Comparison metric |
|----------|---------------------|-------------------|
| `KPM_Tn` | `run_on=:gpu` | ‖T_N(CPU) − T_N(GPU)‖/‖T_N‖ |
| `get_ldos_online` | `run_on=:gpu` | ‖LDOS_CPU − LDOS_GPU‖/‖LDOS_CPU‖ |
| `get_dos_stochastic` | `run_on=:gpu` | ‖DOS_CPU − DOS_GPU‖/‖DOS_CPU‖ |
| `get_bands` | `run_on=:gpu` | ‖A(k,ω)_CPU − A(k,ω)_GPU‖ / ‖A(k,ω)_CPU‖ |
| `mcweeny_purify` | `run_on=:gpu` | ‖ρ_CPU − ρ_GPU‖/‖ρ_CPU‖ |
| `sp2_purify` | `run_on=:gpu` | ‖ρ_CPU − ρ_GPU‖/‖ρ_CPU‖ |
| `get_density` (:mcweeny/:sp2/:kpm) | `run_on=:gpu` | ‖ρ_CPU − ρ_GPU‖/‖ρ_CPU‖ |
| `get_W` | `run_on=:gpu` | \|W_CPU − W_GPU\| |
| `get_C` | `run_on=:gpu` | \|C_CPU − C_GPU\| |
| `evolve_rk4_dm_timedep` | `run_on=:gpu` | max_t ‖ρ(t)_CPU − ρ(t)_GPU‖/‖ρ(t)_CPU‖ |